In [ ]:
from pyspark.sql.functions import col, trim
from delta.tables import DeltaTable
from src.common.spark_utils import standardize_columns
from src.common.quality_events import log_pipeline_failure, quarantine_rejected_rows, reconcile_merge_target

In [ ]:
# Configure the Bronze root for this workspace or job.
dbutils.widgets.text("bronze_root", "abfss://bronze@ubereaststorage.dfs.core.windows.net", "Bronze storage root")
BRONZE_ROOT = dbutils.widgets.get("bronze_root").rstrip("/")


In [ ]:
# Read the Bronze source and log failures before re-raising.
try:
    df_zones_raw = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(f"{BRONZE_ROOT}/csv/zones/")
    )
except Exception as exc:
    log_pipeline_failure(spark, "zones", "source_read", exc)
    raise

source_rows = df_zones_raw.count()
print(f"zones.source_rows={source_rows}")
display(df_zones_raw)
df_zones_raw.printSchema()

In [ ]:
# Standardize columns and capture source quality metrics before publishing.
df_zones = standardize_columns(df_zones_raw)

null_zone_ids = df_zones.filter(
    col("zone_id").isNull() | (trim(col("zone_id").cast("string")) == "")
).count()

duplicate_zone_keys = (
    df_zones
    .filter(col("zone_id").isNotNull())
    .groupBy("zone_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

missing_city_rows = df_zones.filter(
    col("city").isNull() | (trim(col("city")) == "")
).count()
missing_zone_name_rows = df_zones.filter(
    col("zone_name").isNull() | (trim(col("zone_name")) == "")
).count()

latitude_value = col("latitude").cast("double")
longitude_value = col("longitude").cast("double")

invalid_latitude_rows = df_zones.filter(
    latitude_value.isNull()
    | (latitude_value < -90)
    | (latitude_value > 90)
).count()
invalid_longitude_rows = df_zones.filter(
    longitude_value.isNull()
    | (longitude_value < -180)
    | (longitude_value > 180)
).count()

print(f"zones.quality.null_or_blank_zone_id_rows={null_zone_ids}")
print(f"zones.quality.duplicate_zone_id_keys={duplicate_zone_keys}")
print(f"zones.quality.missing_city_rows={missing_city_rows}")
print(f"zones.quality.missing_zone_name_rows={missing_zone_name_rows}")
print(f"zones.quality.invalid_latitude_rows={invalid_latitude_rows}")
print(f"zones.quality.invalid_longitude_rows={invalid_longitude_rows}")

# Quarantine rows violating merge-key or coordinate guardrails before failing.
invalid_zone_rows = df_zones.filter(
    col("zone_id").isNull()
    | (trim(col("zone_id").cast("string")) == "")
    | latitude_value.isNull()
    | (latitude_value < -90)
    | (latitude_value > 90)
    | longitude_value.isNull()
    | (longitude_value < -180)
    | (longitude_value > 180)
)
quarantine_rejected_rows(
    spark, invalid_zone_rows, "zones",
    "invalid_zone_id_or_geographic_coordinates",
)

# Fail before writing if merge keys are unsafe or coordinates are invalid.
# This avoids silently publishing ambiguous zone IDs or unusable map locations.
if null_zone_ids:
    raise ValueError(f"Zones Silver input contains {null_zone_ids} null/blank zone_id rows")
if duplicate_zone_keys:
    raise ValueError(f"Zones Silver input contains {duplicate_zone_keys} duplicate zone_id keys")
if invalid_latitude_rows:
    raise ValueError(f"Zones Silver input contains {invalid_latitude_rows} invalid latitude rows")
if invalid_longitude_rows:
    raise ValueError(f"Zones Silver input contains {invalid_longitude_rows} invalid longitude rows")

prepared_duplicate_keys = (
    df_zones
    .groupBy("zone_id")
    .count()
    .filter(col("count") > 1)
    .count()
)
if prepared_duplicate_keys:
    raise ValueError(
        f"Zones Silver merge input contains {prepared_duplicate_keys} duplicate zone_id keys"
    )

prepared_rows = df_zones.count()
print(f"zones.quality.prepared_rows={prepared_rows}")
print(f"zones.quality.prepared_duplicate_zone_id_keys={prepared_duplicate_keys}")
display(df_zones)

In [ ]:
try:
    table_name = "ubereats_databricks1.silver.zones"
    
    if not spark.catalog.tableExists(table_name):
        (
            df_zones.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )
    else:
        target = DeltaTable.forName(spark, table_name)
        (
            target.alias("t")
            .merge(
                df_zones.alias("s"),
                "t.zone_id = s.zone_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
except Exception as exc:
    # Record publish failures without replacing the original exception.
    failure_record = [{
        "run_id": str(__import__("uuid").uuid4()),
        "dataset_name": "zones",
        "stage": "silver_publish",
        "error_type": type(exc).__name__,
        "error_message": str(exc)[:2000],
        "recorded_at_utc": str(spark.sql("SELECT current_timestamp() AS ts").first()["ts"]),
    }]
    try:
        spark.createDataFrame(failure_record).write.format("delta").mode("append").saveAsTable(
            "ubereats_databricks1.silver.pipeline_failure_events"
        )
    except Exception as log_exc:
        print("zones.failure_log_error=" + str(log_exc)[:500])
    raise

# Reconcile the published Delta target before marking this run successful.
reconcile_merge_target(
    spark=spark,
    source_df=df_zones,
    target_table=table_name,
    dataset_name="zones",
    key_columns=["zone_id"],
    compare_columns=["latitude","longitude","city","zone_name"],
)

# Persist a compact successful-run quality record for later monitoring.
quality_table = "ubereats_databricks1.silver.pipeline_quality_results"
quality_record = [{
    "run_id": str(__import__("uuid").uuid4()),
    "dataset_name": "zones",
    "source_rows": int(source_rows),
    "prepared_rows": int(df_zones.count()),
    "target_rows_after": int(spark.table(table_name).count()),
    "status": "SUCCEEDED",
    "recorded_at_utc": str(spark.sql("SELECT current_timestamp() AS ts").first()["ts"]),
}]
spark.createDataFrame(quality_record).write.format("delta").mode("append").saveAsTable(quality_table)
print("zones.quality.persisted_run_id=" + quality_record[0]["run_id"])
